# Clase 4 — Modelos locales y capacidades para agentes

En esta clase incorporás **Ollama** como provider local sin romper la arquitectura construida en la Clase 3. El objetivo no es declarar un ganador entre cloud y local, sino aprender a seleccionar y enrutar modelos según **capacidad, privacidad, latencia, costo y disponibilidad**.

> **Modelo cloud canónico del curso:** Gemini 3.1 Flash Lite.  
> **Modelo local canónico de esta clase:** Qwen 3.5 4B (`qwen3.5:4b`).

## Resultado de la clase

Al finalizar vas a tener:

- un contrato común para providers cloud y locales;
- un `OllamaProvider` asíncrono con streaming y manejo de errores;
- una prueba explícita de **tool calling nativo** mediante `/api/chat`;
- un benchmark reproducible orientado a capacidades agénticas;
- una política de routing determinística cloud/local;
- tests sin red mediante providers simulados;
- un artefacto `class04_model_benchmark.json`.

---

### Contrato de continuidad

- Las notebooks de las clases 1 y 2 son la **base publicada e inmutable**.
- Esta clase continúa sobre `ai_agent_project/`; no crea un proyecto paralelo.
- Incremento: **Ollama, benchmark y routing cloud/local**.
- El modo predeterminado es local/simulado y no consume APIs por accidente.
- Los modos reales requieren variables de entorno explícitas.

La notebook separa experimentos visibles de módulos reutilizables y deja un checkpoint para la clase siguiente.


## Mapa del laboratorio

1. Recuperar la arquitectura de la Clase 3.
2. Separar modelo, pesos, runtime y provider.
3. Verificar Ollama y seleccionar una ruta de práctica.
4. Implementar el provider local.
5. Comparar capacidades con un benchmark pequeño.
6. Diseñar un router cloud/local.
7. Probar fallos, fallback y decisiones.
8. Guardar el checkpoint.

La notebook funciona sin Ollama ni API key. En ese caso utiliza providers determinísticos y lo informa explícitamente.

### Patrón de trabajo de esta notebook

En cada incremento vamos a seguir la misma secuencia que quedó validada en la Clase 3:

1. **comprender el problema y el contrato**;
2. **construir una versión visible y mínima**;
3. **probar un caso exitoso y un borde relevante**;
4. **persistir la implementación en `src/`**;
5. **importar el módulo generado y volver a verificarlo**.

Los servicios externos permanecen desactivados por defecto. Las celdas que usan APIs, Ollama, Qdrant Server u otros servicios requieren una habilitación explícita.

In [2]:
from pathlib import Path
import asyncio
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import textwrap
import time
from dataclasses import asdict, dataclass
from typing import Any

PROJECT_DIR = Path("ai_agent_project")
SRC_DIR = PROJECT_DIR / "src" / "ai_agent_course"
DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
TESTS_DIR = PROJECT_DIR / "tests"

for directory in (SRC_DIR, DATA_DIR, ARTIFACTS_DIR, TESTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "__init__.py").touch()

if str(PROJECT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "src"))

def write_file(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).strip() + "\n", encoding="utf-8")
    print("✓", path.relative_to(PROJECT_DIR))

def show_table(rows, columns=None):
    try:
        import pandas as pd
        frame = pd.DataFrame(rows)
        if columns:
            frame = frame[columns]
        display(frame)
    except Exception:
        for row in rows:
            print(row)

print(f"Proyecto incremental: {PROJECT_DIR}")

Proyecto incremental: ai_agent_project


### Checkpoint de continuidad

Se verifican los componentes previos antes de sumar el incremento. Un faltante se informa; no se oculta recreando silenciosamente todo el proyecto.

In [3]:
_continuity_root = PROJECT_DIR
_continuity = ['src/ai_agent_course/providers.py', 'artifacts/class03_report.json']
continuity_status = [{"path": p, "available": (_continuity_root / p).exists()} for p in _continuity]
for item in continuity_status:
    print(("✓" if item["available"] else "·"), item["path"])
missing_prerequisites = [item["path"] for item in continuity_status if not item["available"]]
if missing_prerequisites:
    print("Aviso: faltan componentes previos:", missing_prerequisites)
else:
    print("Continuidad verificada.")

✓ src/ai_agent_course/providers.py
✓ artifacts/class03_report.json
Continuidad verificada.


## 1. Entorno y ruta de ejecución

La práctica admite tres rutas:

| Ruta | Requisitos | Uso |
|---|---|---|
| Real local | Ollama activo y un modelo descargado | Medir el equipo del estudiante |
| Real híbrida | Ollama + credencial cloud | Comparar proveedores reales |
| Simulada | Sin servicios externos | Aprender contratos, benchmark y routing |

La simulación no reemplaza la evaluación real de calidad. Sirve para que el laboratorio sea reproducible y para testear la aplicación sin costo.

In [4]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=PROJECT_DIR / ".env", override=True)

RUN_INSTALLS = os.getenv("RUN_INSTALLS", "0") == "1"
PACKAGES = ["httpx>=0.27", "pydantic>=2.0", "pydantic-settings>=2.0", "google-genai>=1.0", "pytest>=8.0"]

if RUN_INSTALLS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *PACKAGES])
else:
    print("Dependencias opcionales:")
    print("python -m pip install " + " ".join(PACKAGES))


OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3.5:4b")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite")
USE_REAL_OLLAMA = os.getenv("USE_REAL_OLLAMA", "0") == "1"
USE_REAL_GEMINI = os.getenv("USE_REAL_GEMINI", "0") == "1"

print(os.getenv("MAX_RETRIES"))
print({
    "ollama_model": OLLAMA_MODEL,
    "cloud_model": GEMINI_MODEL,
    "use_real_ollama": USE_REAL_OLLAMA,
    "use_real_gemini": USE_REAL_GEMINI,
})

Dependencias opcionales:
python -m pip install httpx>=0.27 pydantic>=2.0 pydantic-settings>=2.0 google-genai>=1.0 pytest>=8.0
2
{'ollama_model': 'qwen3.5:4b', 'cloud_model': 'gemini-3.1-flash-lite', 'use_real_ollama': False, 'use_real_gemini': False}


## 2. Recuperar el contrato de provider

La Clase 3 definió una interfaz común. Si el alumno ejecuta esta notebook de forma aislada, se crea una versión mínima compatible. La aplicación no debería conocer detalles de Ollama o Gemini: solo debería depender de `LLMProvider` y `GenerationResult`.

In [5]:
PROVIDERS_PATH = SRC_DIR / "providers.py"

if not PROVIDERS_PATH.exists():
    write_file(PROVIDERS_PATH, r'''
    from __future__ import annotations

    import asyncio
    import time
    from dataclasses import asdict, dataclass
    from typing import AsyncIterator, Protocol

    @dataclass(frozen=True)
    class GenerationResult:
        text: str
        model: str
        provider: str
        latency_ms: float
        input_tokens: int
        output_tokens: int
        finish_reason: str = "stop"

        def to_dict(self) -> dict:
            return asdict(self)

    class LLMProvider(Protocol):
        async def agenerate(
            self,
            prompt: str,
            *,
            system: str | None = None,
            temperature: float = 0.2,
        ) -> GenerationResult: ...

        async def astream(
            self,
            prompt: str,
            *,
            system: str | None = None,
            temperature: float = 0.2,
        ) -> AsyncIterator[str]: ...

    class FakeProvider:
        def __init__(self, model: str = "fake-llm", delay_seconds: float = 0.02):
            self.model = model
            self.delay_seconds = delay_seconds

        async def agenerate(self, prompt: str, *, system=None, temperature=0.2):
            if not prompt.strip():
                raise ValueError("El prompt no puede estar vacío")
            started = time.perf_counter()
            await asyncio.sleep(self.delay_seconds)
            text = f"Respuesta simulada para: {prompt.strip()[:100]}"
            return GenerationResult(
                text=text,
                model=self.model,
                provider="fake",
                latency_ms=round((time.perf_counter() - started) * 1000, 2),
                input_tokens=max(1, len(prompt.split())),
                output_tokens=max(1, len(text.split())),
            )

        async def astream(self, prompt: str, *, system=None, temperature=0.2):
            result = await self.agenerate(prompt, system=system, temperature=temperature)
            for token in result.text.split():
                await asyncio.sleep(self.delay_seconds / 5)
                yield token + " "
    ''')

from ai_agent_course.providers import GenerationResult
print("Contrato disponible:", [field for field in GenerationResult.__dataclass_fields__])

Contrato disponible: ['text', 'model', 'provider', 'latency_ms', 'input_tokens', 'output_tokens', 'finish_reason']


## 3. Modelo, pesos, runtime y provider

Estos conceptos suelen mezclarse:

- **Modelo:** arquitectura y comportamiento aprendido.
- **Pesos:** parámetros concretos que materializan ese modelo.
- **Runtime:** software que carga y ejecuta los pesos.
- **Provider:** adaptador que la aplicación usa para pedir una generación.

Cambiar de runtime no necesariamente cambia el modelo. Cambiar de provider no debería obligar a reescribir toda la aplicación.

In [6]:
concepts = [
    {"concepto": "Modelo", "ejemplo": "Qwen 3.5", "pregunta": "¿Qué capacidades tiene?"},
    {"concepto": "Pesos", "ejemplo": "4B cuantizado", "pregunta": "¿Cuánta memoria requiere?"},
    {"concepto": "Runtime", "ejemplo": "Ollama", "pregunta": "¿Cómo se ejecuta localmente?"},
    {"concepto": "Provider", "ejemplo": "OllamaProvider", "pregunta": "¿Qué contrato ve la app?"},
]
show_table(concepts)

{'concepto': 'Modelo', 'ejemplo': 'Qwen 3.5', 'pregunta': '¿Qué capacidades tiene?'}
{'concepto': 'Pesos', 'ejemplo': '4B cuantizado', 'pregunta': '¿Cuánta memoria requiere?'}
{'concepto': 'Runtime', 'ejemplo': 'Ollama', 'pregunta': '¿Cómo se ejecuta localmente?'}
{'concepto': 'Provider', 'ejemplo': 'OllamaProvider', 'pregunta': '¿Qué contrato ve la app?'}


### Cuantización y memoria: estimación rápida

La cuantización reduce memoria y normalmente también precisión. La siguiente cuenta es una **aproximación pedagógica**: no incluye KV cache, buffers del runtime ni overhead del sistema.

In [7]:
def estimate_weight_memory_gb(parameters_billions: float, bits_per_weight: int) -> float:
    return round(parameters_billions * 1_000_000_000 * bits_per_weight / 8 / 1024**3, 2)

memory_rows = []
for parameters in (4, 8, 14):
    for bits in (4, 8, 16):
        memory_rows.append({
            "parámetros_B": parameters,
            "cuantización_bits": bits,
            "pesos_aprox_GB": estimate_weight_memory_gb(parameters, bits),
        })

show_table(memory_rows)

{'parámetros_B': 4, 'cuantización_bits': 4, 'pesos_aprox_GB': 1.86}
{'parámetros_B': 4, 'cuantización_bits': 8, 'pesos_aprox_GB': 3.73}
{'parámetros_B': 4, 'cuantización_bits': 16, 'pesos_aprox_GB': 7.45}
{'parámetros_B': 8, 'cuantización_bits': 4, 'pesos_aprox_GB': 3.73}
{'parámetros_B': 8, 'cuantización_bits': 8, 'pesos_aprox_GB': 7.45}
{'parámetros_B': 8, 'cuantización_bits': 16, 'pesos_aprox_GB': 14.9}
{'parámetros_B': 14, 'cuantización_bits': 4, 'pesos_aprox_GB': 6.52}
{'parámetros_B': 14, 'cuantización_bits': 8, 'pesos_aprox_GB': 13.04}
{'parámetros_B': 14, 'cuantización_bits': 16, 'pesos_aprox_GB': 26.08}


## 4. Verificar Ollama

No alcanza con que el comando exista: también debe responder la API local. La celda distingue instalación, daemon y modelos disponibles.

In [8]:
def inspect_ollama(base_url: str = "http://localhost:11434") -> dict[str, Any]:
    status: dict[str, Any] = {
        "command_available": shutil.which("ollama") is not None,
        "api_available": False,
        "version": None,
        "models": [],
        "reason": None,
    }
    if status["command_available"]:
        try:
            status["version"] = subprocess.check_output(
                ["ollama", "--version"], text=True, timeout=5
            ).strip()
        except Exception as exc:
            status["reason"] = f"No se pudo leer la versión: {exc}"

    try:
        import httpx
        response = httpx.get(f"{base_url}/api/tags", timeout=2)
        response.raise_for_status()
        status["api_available"] = True
        status["models"] = [item.get("name") for item in response.json().get("models", [])]
    except Exception as exc:
        status["reason"] = status["reason"] or f"API local no disponible: {type(exc).__name__}"

    return status

ollama_status = inspect_ollama()
ollama_status

{'command_available': True,
 'api_available': True,
 'version': 'ollama version is 0.32.14',
 'models': ['qwen3.5:4b'],
 'reason': None}

In [9]:
available_models = set(ollama_status["models"])
selected_model_available = OLLAMA_MODEL in available_models

print("Modelo real?", USE_REAL_OLLAMA)
route = (
    "real-local"
    if USE_REAL_OLLAMA
    and ollama_status["api_available"]
    and selected_model_available
    else "simulada"
)

print("Ruta activa:", route)
print("Modelo local seleccionado:", OLLAMA_MODEL)

if ollama_status["api_available"] and not selected_model_available:
    print(f"⚠ El modelo {OLLAMA_MODEL!r} no está descargado.")
    print(f"  Ejecutá: ollama pull {OLLAMA_MODEL}")

if USE_REAL_OLLAMA and not ollama_status["api_available"]:
    print("⚠ USE_REAL_OLLAMA=1, pero Ollama no responde. Se usará el fallback simulado.")
elif USE_REAL_OLLAMA and not selected_model_available:
    print("⚠ USE_REAL_OLLAMA=1, pero falta el modelo seleccionado. Se usará el fallback simulado.")


Modelo real? False
Ruta activa: simulada
Modelo local seleccionado: qwen3.5:4b


## 5. Configuración segura

El modelo cloud correcto es **Gemini 3.1 Flash Lite**. La credencial nunca se imprime ni se escribe dentro de la notebook.

### 5.1 Comprender y probar la configuración antes de crear el módulo

`ModelSettings` concentra las decisiones que cambian entre una ejecución local y una cloud. La validación debe ocurrir antes de construir un provider: un modo real mal configurado no debería fallar recién durante la llamada de red.

In [10]:
from pathlib import Path
from pydantic import Field, SecretStr, ValidationError, model_validator
from pydantic_settings import BaseSettings, SettingsConfigDict

class ModelSettingsPreview(BaseSettings):
    model_config = SettingsConfigDict(extra="ignore", case_sensitive=False)

    gemini_model: str = "gemini-3.1-flash-lite"
    ollama_model: str = "qwen3.5:4b"
    ollama_base_url: str = "http://localhost:11434"
    timeout_seconds: float = Field(default=30.0, gt=0)
    use_real_gemini: bool = False
    use_real_ollama: bool = False
    gemini_api_key: SecretStr | None = None

    @model_validator(mode="after")
    def validate_cloud(self):
        if self.use_real_gemini and self.gemini_api_key is None:
            raise ValueError("USE_REAL_GEMINI=1 requiere GEMINI_API_KEY")
        return self

    def safe_dict(self) -> dict:
        data = self.model_dump(exclude={"gemini_api_key"})
        data["gemini_api_key"] = "***configurada***" if self.gemini_api_key else None
        return data

preview_settings = ModelSettingsPreview(
    gemini_model="gemini-3.1-flash-lite",
    ollama_model="qwen3.5:4b",
    use_real_gemini=False,
    use_real_ollama=False,
    gemini_api_key=None,
)
assert preview_settings.gemini_model == "gemini-3.1-flash-lite"
assert preview_settings.use_real_gemini is False

validation_results = {}
for case_name, factory in {
    "timeout_invalido": lambda: ModelSettingsPreview(timeout_seconds=0),
    "gemini_sin_key": lambda: ModelSettingsPreview(
        use_real_gemini=True, gemini_api_key=None
    ),
}.items():
    try:
        factory()
        validation_results[case_name] = "NO detectado"
    except ValidationError as exc:
        validation_results[case_name] = exc.errors(include_url=False)[0]["msg"]

secret_preview = ModelSettingsPreview(gemini_api_key="clave-demo").safe_dict()
assert secret_preview["gemini_api_key"] == "***configurada***"

{
    "defaults": preview_settings.safe_dict(),
    "validaciones": validation_results,
    "secreto_enmascarado": secret_preview,
}

{'defaults': {'gemini_model': 'gemini-3.1-flash-lite',
  'ollama_model': 'qwen3.5:4b',
  'ollama_base_url': 'http://localhost:11434',
  'timeout_seconds': 30.0,
  'use_real_gemini': False,
  'use_real_ollama': False,
  'gemini_api_key': None},
 'validaciones': {'timeout_invalido': 'Input should be greater than 0',
  'gemini_sin_key': 'Value error, USE_REAL_GEMINI=1 requiere GEMINI_API_KEY'},
 'secreto_enmascarado': {'gemini_model': 'gemini-3.1-flash-lite',
  'ollama_model': 'qwen3.5:4b',
  'ollama_base_url': 'http://localhost:11434',
  'timeout_seconds': 30.0,
  'use_real_gemini': False,
  'use_real_ollama': False,
  'gemini_api_key': '***configurada***'}}

### 5.2 Persistir la implementación validada

Ahora trasladamos el contrato probado a `model_settings.py`. El archivo queda visible en la celda, no encapsulado dentro de un string.

In [11]:
%%writefile ai_agent_project/src/ai_agent_course/model_settings.py
from __future__ import annotations

from pathlib import Path

from pydantic import Field, SecretStr, model_validator
from pydantic_settings import BaseSettings, SettingsConfigDict


class ModelSettings(BaseSettings):
    """Configuración de modelos cloud y locales para la Clase 4."""

    model_config = SettingsConfigDict(
        extra="ignore",
        case_sensitive=False,
        env_file_encoding="utf-8",
    )

    gemini_model: str = "gemini-3.1-flash-lite"
    ollama_model: str = "qwen3.5:4b"
    ollama_base_url: str = "http://localhost:11434"
    timeout_seconds: float = Field(default=30.0, gt=0)

    use_real_gemini: bool = False
    use_real_ollama: bool = False
    gemini_api_key: SecretStr | None = None

    @model_validator(mode="after")
    def validate_cloud(self):
        if self.use_real_gemini and self.gemini_api_key is None:
            raise ValueError("USE_REAL_GEMINI=1 requiere GEMINI_API_KEY")
        return self

    @classmethod
    def from_env(cls, env_path: str | Path | None = None):
        path = Path(env_path) if env_path is not None else None
        values = {"_env_file": path} if path is not None and path.exists() else {}
        return cls(**values)

    def api_key_value(self) -> str | None:
        return self.gemini_api_key.get_secret_value() if self.gemini_api_key else None

    def safe_dict(self) -> dict:
        data = self.model_dump(exclude={"gemini_api_key"})
        data["gemini_api_key"] = "***configurada***" if self.gemini_api_key else None
        return data


Overwriting ai_agent_project/src/ai_agent_course/model_settings.py


In [12]:
# Actualizamos .env.example sin borrar las variables agregadas en clases anteriores.
env_path = PROJECT_DIR / ".env.example"
existing_lines = env_path.read_text(encoding="utf-8").splitlines() if env_path.exists() else []

new_env_values = {
    "GEMINI_MODEL": "gemini-3.1-flash-lite",
    "USE_REAL_GEMINI": "0",
    "OLLAMA_MODEL": "qwen3.5:4b",
    "OLLAMA_BASE_URL": "http://localhost:11434",
    "USE_REAL_OLLAMA": "0",
    "TIMEOUT_SECONDS": "30",
}

updated_lines = []
updated_keys = set()
for line in existing_lines:
    stripped = line.lstrip()
    if "=" not in line or stripped.startswith("#"):
        updated_lines.append(line)
        continue

    key = line.split("=", 1)[0].strip()
    if key in new_env_values:
        updated_lines.append(f"{key}={new_env_values[key]}")
        updated_keys.add(key)
    else:
        updated_lines.append(line)

for key, value in new_env_values.items():
    if key not in updated_keys:
        updated_lines.append(f"{key}={value}")

env_path.write_text("\n".join(updated_lines).rstrip() + "\n", encoding="utf-8")
print(env_path.read_text(encoding="utf-8"))


# Entorno de ejecución
APP_ENV=development

# Modo seguro predeterminado: no realiza llamadas reales.
# Cambiar a 1 solamente después de configurar GEMINI_API_KEY.
USE_REAL_GEMINI=0
GEMINI_API_KEY=
DEFAULT_MODEL=gemini-3.1-flash-lite

# Resiliencia y concurrencia
REQUEST_TIMEOUT_SECONDS=20
MAX_RETRIES=2
RETRY_BASE_DELAY_SECONDS=0.25
MAX_CONCURRENCY=3
HISTORY_MAX_TURNS=6

# Valores educativos para estimación de costos.
# Revisarlos antes de usarlos para decisiones económicas reales.
INPUT_USD_PER_MILLION=0.25
OUTPUT_USD_PER_MILLION=1.50
GEMINI_MODEL=gemini-3.1-flash-lite
OLLAMA_MODEL=qwen3.5:4b
OLLAMA_BASE_URL=http://localhost:11434
USE_REAL_OLLAMA=0
TIMEOUT_SECONDS=30



In [13]:
import importlib
import ai_agent_course.model_settings as model_settings_module
importlib.reload(model_settings_module)

settings = model_settings_module.ModelSettings.from_env(PROJECT_DIR / ".env")
assert settings.gemini_model == "gemini-3.1-flash-lite"
settings.safe_dict()

{'gemini_model': 'gemini-3.1-flash-lite',
 'ollama_model': 'qwen3.5:4b',
 'ollama_base_url': 'http://localhost:11434',
 'timeout_seconds': 30.0,
 'use_real_gemini': False,
 'use_real_ollama': False,
 'gemini_api_key': '***configurada***'}

## 6. `OllamaProvider` asíncrono

El provider:

- respeta el mismo contrato de generación que el provider cloud;
- usa `httpx.AsyncClient`;
- soporta generación completa y streaming mediante `/api/generate`;
- expone una operación separada para **tool calling nativo** mediante `/api/chat`;
- valida respuestas vacías;
- traduce errores HTTP a excepciones comprensibles;
- utiliza un fallback explícito cuando el servicio no está activo.

La operación de tools se mantiene separada porque `GenerationResult` representa texto generado, mientras que una solicitud de herramienta devuelve una estructura diferente: `tool_calls`.


### 6.1 Comprender el adaptador antes de persistirlo

El adapter de Ollama no decide qué tarea ejecutar: traduce el contrato común del curso a los payloads HTTP de Ollama. Antes de incorporar la red, verificamos responsabilidades separadas:

- un provider simulado devuelve el mismo `GenerationResult` que el resto del proyecto;
- `/api/generate` conserva modelo, prompt, system y temperatura;
- `/api/chat` recibe mensajes y una definición estructurada de `tools`.

**Importante:** pedirle al modelo que escriba un JSON con el nombre de una tool no equivale a tool calling nativo. La prueba nativa debe inspeccionar el campo `message.tool_calls` de la respuesta.


In [14]:
from ai_agent_project.src.ai_agent_course.providers import GenerationResult
import asyncio
import time

class SimulatedAgentProviderPreview:
    def __init__(self, name: str, model: str, latency_ms: float = 1.0):
        self.name = name
        self.model = model
        self.latency_ms = latency_ms

    async def agenerate(self, prompt: str, *, system=None, temperature=0.2):
        if not prompt.strip():
            raise ValueError("El prompt no puede estar vacío")
        started = time.perf_counter()
        await asyncio.sleep(self.latency_ms / 1000)
        text = f"Respuesta simulada: {prompt}"
        return GenerationResult(
            text=text,
            model=self.model,
            provider=self.name,
            latency_ms=round((time.perf_counter() - started) * 1000, 2),
            input_tokens=len(prompt.split()),
            output_tokens=len(text.split()),
        )

def build_ollama_generate_payload(
    model: str,
    prompt: str,
    system: str | None,
    temperature: float,
) -> dict:
    if not prompt.strip():
        raise ValueError("El prompt no puede estar vacío")
    return {
        "model": model,
        "prompt": prompt,
        "system": system or "",
        "stream": False,
        "options": {"temperature": temperature},
    }

def build_ollama_chat_payload(
    model: str,
    prompt: str,
    tools: list[dict],
    system: str | None,
    temperature: float,
) -> dict:
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    return {
        "model": model,
        "messages": messages,
        "tools": tools,
        "stream": False,
        "options": {"temperature": temperature},
    }

preview_provider = SimulatedAgentProviderPreview("local-preview", "qwen3.5:4b")
preview_result = await preview_provider.agenerate(prompt="Explicá qué es un provider")
preview_generate_payload = build_ollama_generate_payload(model="qwen3.5:4b", prompt="Hola",
                                                            system="Sé breve",temperature=0.2)
preview_chat_payload = build_ollama_chat_payload(
    model="qwen3.5:4b",
    prompt="Consultá la política de vacaciones.",
    tools=[{
        "type": "function",
        "function": {
            "name": "buscar_politica",
            "description": "Busca una política interna por tema.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
            },
        },
    }],
    system="Usá una herramienta cuando corresponda.",
    temperature=0.0,
)

assert preview_result.provider == "local-preview"
assert preview_generate_payload["options"]["temperature"] == 0.2
assert preview_chat_payload["tools"][0]["function"]["name"] == "buscar_politica"
{
    "result": preview_result.to_dict(),
    "generate_payload": preview_generate_payload,
    "chat_payload": preview_chat_payload,
}


{'result': {'text': 'Respuesta simulada: Explicá qué es un provider',
  'model': 'qwen3.5:4b',
  'provider': 'local-preview',
  'latency_ms': 1.16,
  'input_tokens': 5,
  'output_tokens': 7,
  'finish_reason': 'stop'},
 'generate_payload': {'model': 'qwen3.5:4b',
  'prompt': 'Hola',
  'system': 'Sé breve',
  'stream': False,
  'options': {'temperature': 0.2}},
 'chat_payload': {'model': 'qwen3.5:4b',
  'messages': [{'role': 'system',
    'content': 'Usá una herramienta cuando corresponda.'},
   {'role': 'user', 'content': 'Consultá la política de vacaciones.'}],
  'tools': [{'type': 'function',
    'function': {'name': 'buscar_politica',
     'description': 'Busca una política interna por tema.',
     'parameters': {'type': 'object',
      'properties': {'query': {'type': 'string'}},
      'required': ['query']}}}],
  'stream': False,
  'options': {'temperature': 0.0}}}

### 6.2 Persistir los providers locales

La implementación completa agrega HTTP asíncrono, streaming, detección de respuestas vacías y fallback explícito. La llamada real sigue desactivada si Ollama no está disponible.

In [15]:
%%writefile ai_agent_project/src/ai_agent_course/local_models.py
from __future__ import annotations

import asyncio
import json
import time
from typing import AsyncIterator

import httpx

from .providers import GenerationResult


class LocalProviderError(RuntimeError):
    pass


class SimulatedAgentProvider:
    def __init__(
        self,
        *,
        name: str,
        model: str,
        latency_ms: float,
        supports_tools: bool,
        structured_reliability: float,
    ):
        self.name = name
        self.model = model
        self.latency_ms = latency_ms
        self.supports_tools = supports_tools
        self.structured_reliability = structured_reliability

    def _response_for(self, prompt: str) -> str:
        lowered = prompt.lower()
        if "exactamente 7 palabras" in lowered:
            return "Agente decide acciones usando herramientas y contexto"
        if "seleccioná una herramienta" in lowered:
            return (
                '{"tool":"buscar_politica","arguments":{"query":"vacaciones"}}'
                if self.supports_tools
                else "No puedo seleccionar herramientas de forma confiable."
            )
        if "solo json" in lowered:
            if self.structured_reliability < 0.90:
                return "category=rrhh; confidence=0.93"
            return '{"category":"rrhh","confidence":0.93}'
        if "resumí" in lowered:
            return "Resumen: la política requiere registro, revisión y aprobación."
        return f"Respuesta de {self.name}: {prompt[:100]}"

    async def agenerate(self, prompt: str, *, system=None, temperature=0.2):
        if not prompt.strip():
            raise ValueError("El prompt no puede estar vacío")
        started = time.perf_counter()
        await asyncio.sleep(self.latency_ms / 1000)
        text = self._response_for(prompt)
        return GenerationResult(
            text=text,
            model=self.model,
            provider=self.name,
            latency_ms=round((time.perf_counter() - started) * 1000, 2),
            input_tokens=max(1, len(prompt.split())),
            output_tokens=max(1, len(text.split())),
        )

    async def astream(self, prompt: str, *, system=None, temperature=0.2):
        result = await self.agenerate(prompt, system=system, temperature=temperature)
        for token in result.text.split():
            await asyncio.sleep(0.005)
            yield token + " "

    async def achat_with_tools(
        self,
        prompt: str,
        *,
        tools: list[dict],
        system: str | None = None,
        temperature: float = 0.0,
    ) -> dict:
        if not prompt.strip():
            raise ValueError("El prompt no puede estar vacío")
        if not tools:
            raise ValueError("Se requiere al menos una tool")

        started = time.perf_counter()
        await asyncio.sleep(self.latency_ms / 1000)
        tool_calls = []
        if self.supports_tools:
            function = tools[0].get("function", {})
            tool_calls = [{
                "function": {
                    "name": function.get("name", "buscar_politica"),
                    "arguments": {"query": "vacaciones"},
                }
            }]

        return {
            "content": "" if tool_calls else "No se solicitó una herramienta.",
            "model": self.model,
            "provider": self.name,
            "latency_ms": round((time.perf_counter() - started) * 1000, 2),
            "tool_calls": tool_calls,
        }


class OllamaProvider:
    def __init__(
        self,
        model: str = "qwen3.5:4b",
        base_url: str = "http://localhost:11434",
        timeout_seconds: float = 30.0,
        fallback=None,
    ):
        self.model = model
        self.base_url = base_url.rstrip("/")
        self.timeout_seconds = timeout_seconds
        self.fallback = fallback

    async def agenerate(self, prompt: str, *, system=None, temperature=0.2):
        if not prompt.strip():
            raise ValueError("El prompt no puede estar vacío")

        payload = {
            "model": self.model,
            "prompt": prompt,
            "system": system or "",
            "stream": False,
            "options": {"temperature": temperature},
        }
        started = time.perf_counter()
        try:
            async with httpx.AsyncClient(timeout=self.timeout_seconds) as client:
                response = await client.post(f"{self.base_url}/api/generate", json=payload)
                response.raise_for_status()
                data = response.json()
        except (httpx.HTTPError, ValueError) as exc:
            if self.fallback is not None:
                return await self.fallback.agenerate(
                    prompt, system=system, temperature=temperature
                )
            raise LocalProviderError(f"Ollama no disponible: {exc}") from exc

        text = str(data.get("response", "")).strip()
        if not text:
            raise LocalProviderError("Ollama devolvió una respuesta vacía")

        return GenerationResult(
            text=text,
            model=self.model,
            provider="ollama",
            latency_ms=round((time.perf_counter() - started) * 1000, 2),
            input_tokens=int(data.get("prompt_eval_count", 0) or 0),
            output_tokens=int(data.get("eval_count", 0) or 0),
        )

    async def astream(self, prompt: str, *, system=None, temperature=0.2) -> AsyncIterator[str]:
        if not prompt.strip():
            raise ValueError("El prompt no puede estar vacío")

        payload = {
            "model": self.model,
            "prompt": prompt,
            "system": system or "",
            "stream": True,
            "options": {"temperature": temperature},
        }
        try:
            async with httpx.AsyncClient(timeout=self.timeout_seconds) as client:
                async with client.stream(
                    "POST", f"{self.base_url}/api/generate", json=payload
                ) as response:
                    response.raise_for_status()
                    emitted = False
                    async for line in response.aiter_lines():
                        if not line:
                            continue
                        data = json.loads(line)
                        token = str(data.get("response", ""))
                        if token:
                            emitted = True
                            yield token
                    if not emitted:
                        raise LocalProviderError("El stream no emitió contenido")
        except (httpx.HTTPError, ValueError, json.JSONDecodeError) as exc:
            if self.fallback is None:
                raise LocalProviderError(f"Falló el streaming local: {exc}") from exc
            async for token in self.fallback.astream(
                prompt, system=system, temperature=temperature
            ):
                yield token

    async def achat_with_tools(
        self,
        prompt: str,
        *,
        tools: list[dict],
        system: str | None = None,
        temperature: float = 0.0,
    ) -> dict:
        """Solicita tool calling nativo mediante Ollama /api/chat.

        Esta operación detecta la intención de llamar una herramienta, pero no la
        ejecuta. La ejecución y el loop de observación se trabajan más adelante.
        """
        if not prompt.strip():
            raise ValueError("El prompt no puede estar vacío")
        if not tools:
            raise ValueError("Se requiere al menos una tool")

        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})

        payload = {
            "model": self.model,
            "messages": messages,
            "tools": tools,
            "stream": False,
            "options": {"temperature": temperature},
        }

        started = time.perf_counter()
        try:
            async with httpx.AsyncClient(timeout=self.timeout_seconds) as client:
                response = await client.post(f"{self.base_url}/api/chat", json=payload)
                response.raise_for_status()
                data = response.json()
        except (httpx.HTTPError, ValueError) as exc:
            if self.fallback is not None and hasattr(self.fallback, "achat_with_tools"):
                return await self.fallback.achat_with_tools(
                    prompt,
                    tools=tools,
                    system=system,
                    temperature=temperature,
                )
            raise LocalProviderError(f"Falló el tool calling local: {exc}") from exc

        message = data.get("message", {})
        return {
            "content": str(message.get("content", "")).strip(),
            "model": str(data.get("model", self.model)),
            "provider": "ollama",
            "latency_ms": round((time.perf_counter() - started) * 1000, 2),
            "input_tokens": int(data.get("prompt_eval_count", 0) or 0),
            "output_tokens": int(data.get("eval_count", 0) or 0),
            "tool_calls": message.get("tool_calls", []) or [],
        }


Overwriting ai_agent_project/src/ai_agent_course/local_models.py


In [16]:
from ai_agent_project.src.ai_agent_course.local_models import OllamaProvider, SimulatedAgentProvider

local_simulated = SimulatedAgentProvider(
    name="ollama-simulado",
    model=OLLAMA_MODEL,
    latency_ms=35,
    supports_tools=True,
    structured_reliability=0.80,
)
local_provider = (
    OllamaProvider(
        model=OLLAMA_MODEL,
        base_url=settings.ollama_base_url,
        timeout_seconds=settings.timeout_seconds,
        fallback=local_simulated,
    )
    if route == "real-local"
    else local_simulated
)
print ("Route activa:", route)
print("Provider local activo:", type(local_provider).__name__)

sample = await local_provider.agenerate(
    "Explicá en una oración qué aporta un modelo local."
)
sample.to_dict()

Route activa: simulada
Provider local activo: SimulatedAgentProvider


{'text': 'Respuesta de ollama-simulado: Explicá en una oración qué aporta un modelo local.',
 'model': 'qwen3.5:4b',
 'provider': 'ollama-simulado',
 'latency_ms': 35.16,
 'input_tokens': 9,
 'output_tokens': 12,
 'finish_reason': 'stop'}

### Streaming

El tiempo hasta el primer fragmento puede ser más importante que la latencia total percibida por el usuario. La celda acumula el texto y mide ambos tiempos.

In [17]:
from ai_agent_project.src.ai_agent_course.providers import LLMProvider

async def consume_stream(provider: LLMProvider, prompt: str) -> dict:
    started = time.perf_counter()
    first_chunk_ms = None
    parts = []
    async for chunk in provider.astream(prompt):
        if first_chunk_ms is None:
            first_chunk_ms = (time.perf_counter() - started) * 1000
        parts.append(chunk)
    return {
        "text": "".join(parts).strip(),
        "chunks": len(parts),
        "first_chunk_ms": round(first_chunk_ms or 0.0, 2),
        "total_ms": round((time.perf_counter() - started) * 1000, 2),
    }

stream_result = await consume_stream(
    local_provider,
    "Resumí por qué una app debe abstraer el provider."
)
stream_result

{'text': 'Resumen: la política requiere registro, revisión y aprobación.',
 'chunks': 8,
 'first_chunk_ms': 41.8,
 'total_ms': 81.55}

### Tool calling nativo: medir la capacidad sin construir todavía el agente

Esta prueba utiliza `/api/chat` y envía una definición real de `tools`. El resultado esperado no es un JSON escrito libremente por el modelo, sino una estructura en `message.tool_calls`.

En esta clase solo verificamos que el modelo pueda **seleccionar y parametrizar** una herramienta. La ejecución de la función, la devolución de la observación y el loop completo se desarrollan en la Clase 9.


In [18]:
POLICY_TOOL = {
    "type": "function",
    "function": {
        "name": "buscar_politica",
        "description": "Busca una política interna por tema.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Tema de la política que se desea consultar.",
                }
            },
            "required": ["query"],
        },
    },
}

native_tool_result = await local_provider.achat_with_tools(
    "Necesito consultar la política de vacaciones. Usá la herramienta disponible.",
    tools=[POLICY_TOOL],
    system="Seleccioná una herramienta cuando sea necesaria. No inventes resultados.",
    temperature=0.0,
)

native_tool_calls = native_tool_result.get("tool_calls", [])
native_tool_check = {
    "provider": native_tool_result.get("provider"),
    "model": native_tool_result.get("model"),
    "tool_call_detected": bool(native_tool_calls),
    "tool_name": (
        native_tool_calls[0].get("function", {}).get("name")
        if native_tool_calls else None
    ),
    "arguments": (
        native_tool_calls[0].get("function", {}).get("arguments")
        if native_tool_calls else None
    ),
    "latency_ms": native_tool_result.get("latency_ms"),
}

show_table([native_tool_check])


{'provider': 'ollama-simulado', 'model': 'qwen3.5:4b', 'tool_call_detected': True, 'tool_name': 'buscar_politica', 'arguments': {'query': 'vacaciones'}, 'latency_ms': 36.1}


## 7. Benchmark orientado a agentes

No medimos “inteligencia general”. Evaluamos tareas que después importan en agentes:

- cumplimiento exacto de instrucciones;
- structured output;
- selección de una tool expresada como JSON;
- resumen breve;
- latencia.

El caso de selección por prompt mide **obediencia y formato**, no tool calling nativo. La capacidad nativa se registró por separado inspeccionando `message.tool_calls`.

Cada caso tiene un evaluador determinístico. En producción convendría repetir varias veces, registrar versiones y sumar un dataset más representativo.


In [19]:
BENCHMARK_CASES = [
    {
        "id": "instruction_7_words",
        "capability": "instruction_following",
        "prompt": "Respondé con exactamente 7 palabras: qué es un agente.",
    },
    {
        "id": "structured_json",
        "capability": "structured_output",
        "prompt": 'Devolvé SOLO JSON con category y confidence para: "necesito vacaciones".',
    },
    {
        "id": "tool_selection_prompted",
        "capability": "prompted_tool_selection",
        "prompt": (
            'Seleccioná una herramienta. Devolvé SOLO JSON con tool y arguments. '
            'Herramienta disponible: buscar_politica(query). Consulta: vacaciones.'
        ),
    },
    {
        "id": "short_summary",
        "capability": "summarization",
        "prompt": (
            "Resumí en una oración: la solicitud se registra, el líder revisa "
            "cobertura y después confirma o rechaza."
        ),
    },
]
show_table(BENCHMARK_CASES, ["id", "capability", "prompt"])

{'id': 'instruction_7_words', 'capability': 'instruction_following', 'prompt': 'Respondé con exactamente 7 palabras: qué es un agente.'}
{'id': 'structured_json', 'capability': 'structured_output', 'prompt': 'Devolvé SOLO JSON con category y confidence para: "necesito vacaciones".'}
{'id': 'tool_selection_prompted', 'capability': 'prompted_tool_selection', 'prompt': 'Seleccioná una herramienta. Devolvé SOLO JSON con tool y arguments. Herramienta disponible: buscar_politica(query). Consulta: vacaciones.'}
{'id': 'short_summary', 'capability': 'summarization', 'prompt': 'Resumí en una oración: la solicitud se registra, el líder revisa cobertura y después confirma o rechaza.'}


In [20]:
def evaluate_case(case: dict, text: str) -> tuple[bool, str]:
    if case["id"] == "instruction_7_words":
        words = text.strip().rstrip(".").split()
        return len(words) == 7, f"{len(words)} palabras"

    if case["id"] == "structured_json":
        try:
            data = json.loads(text)
            ok = {"category", "confidence"} <= data.keys()
            return ok, "JSON válido" if ok else "faltan campos"
        except json.JSONDecodeError:
            return False, "JSON inválido"

    if case["id"] == "tool_selection_prompted":
        try:
            data = json.loads(text)
            ok = data.get("tool") == "buscar_politica" and isinstance(
                data.get("arguments"), dict
            )
            return ok, "tool válida" if ok else "selección incorrecta"
        except json.JSONDecodeError:
            return False, "JSON inválido"

    if case["id"] == "short_summary":
        return len(text.split()) <= 18, f"{len(text.split())} palabras"

    return False, "sin evaluador"

In [21]:
cloud_simulated = SimulatedAgentProvider(
    name="gemini-simulado",
    model=GEMINI_MODEL,
    latency_ms=70,
    supports_tools=True,
    structured_reliability=0.98,
)

cloud_provider = cloud_simulated
if settings.use_real_gemini and settings.api_key_value():
    try:
        from ai_agent_course.providers import GeminiProvider
        cloud_provider = GeminiProvider(
            settings.api_key_value(),
            settings.gemini_model,
        )
        print("Cloud provider real habilitado:", settings.gemini_model)
    except Exception as exc:
        print(
            "⚠ No se pudo habilitar Gemini real; "
            f"se mantiene la simulación: {type(exc).__name__}"
        )

providers = {
    "local": local_provider,
    "cloud": cloud_provider,
}

BENCHMARK_REPETITIONS = 3

async def run_benchmark(
    providers: dict,
    cases: list[dict],
    repetitions: int = BENCHMARK_REPETITIONS,
) -> list[dict]:
    if repetitions < 1:
        raise ValueError("repetitions debe ser al menos 1")

    rows = []
    for provider_name, provider in providers.items():
        for case in cases:
            for repetition in range(1, repetitions + 1):
                result = await provider.agenerate(case["prompt"], temperature=0)
                passed, detail = evaluate_case(case, result.text)
                rows.append({
                    "provider": provider_name,
                    "model": result.model,
                    "case": case["id"],
                    "capability": case["capability"],
                    "repetition": repetition,
                    "passed": passed,
                    "detail": detail,
                    "latency_ms": result.latency_ms,
                    "preview": result.text[:90],
                })
    return rows

benchmark_rows = await run_benchmark(providers, BENCHMARK_CASES)

show_table(
    benchmark_rows,
    ["provider", "model", "case", "repetition", "passed", "detail", "latency_ms", "preview"],
)

{'provider': 'local', 'model': 'qwen3.5:4b', 'case': 'instruction_7_words', 'capability': 'instruction_following', 'repetition': 1, 'passed': True, 'detail': '7 palabras', 'latency_ms': 36.11, 'preview': 'Agente decide acciones usando herramientas y contexto'}
{'provider': 'local', 'model': 'qwen3.5:4b', 'case': 'instruction_7_words', 'capability': 'instruction_following', 'repetition': 2, 'passed': True, 'detail': '7 palabras', 'latency_ms': 36.11, 'preview': 'Agente decide acciones usando herramientas y contexto'}
{'provider': 'local', 'model': 'qwen3.5:4b', 'case': 'instruction_7_words', 'capability': 'instruction_following', 'repetition': 3, 'passed': True, 'detail': '7 palabras', 'latency_ms': 36.12, 'preview': 'Agente decide acciones usando herramientas y contexto'}
{'provider': 'local', 'model': 'qwen3.5:4b', 'case': 'structured_json', 'capability': 'structured_output', 'repetition': 1, 'passed': False, 'detail': 'JSON inválido', 'latency_ms': 36.14, 'preview': 'category=rrhh; c

In [22]:
summary_rows = []
for provider_name in providers:
    own_rows = [row for row in benchmark_rows if row["provider"] == provider_name]
    latencies = [row["latency_ms"] for row in own_rows]
    prompted_tool_rows = [
        row for row in own_rows if row["case"] == "tool_selection_prompted"
    ]
    summary_rows.append({
        "provider": provider_name,
        "model": own_rows[0]["model"],
        "repetitions_per_case": BENCHMARK_REPETITIONS,
        "pass_rate": round(sum(row["passed"] for row in own_rows) / len(own_rows), 2),
        "avg_latency_ms": round(sum(latencies) / len(latencies), 2),
        "latency_spread_ms": round(max(latencies) - min(latencies), 2),
        "prompted_tool_case_passed": all(row["passed"] for row in prompted_tool_rows),
    })

show_table(summary_rows)


{'provider': 'local', 'model': 'qwen3.5:4b', 'repetitions_per_case': 3, 'pass_rate': 0.75, 'avg_latency_ms': 36.13, 'latency_spread_ms': 0.14, 'prompted_tool_case_passed': True}
{'provider': 'cloud', 'model': 'gemini-3.1-flash-lite', 'repetitions_per_case': 3, 'pass_rate': 1.0, 'avg_latency_ms': 71.18, 'latency_spread_ms': 1.96, 'prompted_tool_case_passed': True}


### Interpretar sin simplificar de más

Una tabla de cuatro casos no demuestra que un modelo sea “mejor”. Sí permite detectar incompatibilidades tempranas. Por ejemplo, un modelo local puede producir correctamente un JSON de selección y aun así fallar al usar el protocolo nativo de tools, o puede solicitar la tool correcta pero construir argumentos deficientes.

Por eso conservamos dos evidencias diferentes:

1. **benchmark por prompt:** cumplimiento y formato;
2. **prueba nativa:** presencia y contenido de `message.tool_calls`.


In [23]:
decision_matrix = [
    {
        "criterio": "Privacidad",
        "local": "Datos pueden permanecer en el equipo",
        "cloud": "Depende de políticas y contrato",
    },
    {
        "criterio": "Tool calling",
        "local": "Qwen 3.5 lo soporta; debe medirse su confiabilidad",
        "cloud": "Suele tener soporte más maduro",
    },
    {
        "criterio": "Escalabilidad",
        "local": "Limitada por hardware propio",
        "cloud": "Elasticidad administrada",
    },
    {
        "criterio": "Costo marginal",
        "local": "Infraestructura y energía",
        "cloud": "Pago por uso",
    },
    {
        "criterio": "Disponibilidad offline",
        "local": "Sí",
        "cloud": "No",
    },
]
show_table(decision_matrix)

{'criterio': 'Privacidad', 'local': 'Datos pueden permanecer en el equipo', 'cloud': 'Depende de políticas y contrato'}
{'criterio': 'Tool calling', 'local': 'Qwen 3.5 lo soporta; debe medirse su confiabilidad', 'cloud': 'Suele tener soporte más maduro'}
{'criterio': 'Escalabilidad', 'local': 'Limitada por hardware propio', 'cloud': 'Elasticidad administrada'}
{'criterio': 'Costo marginal', 'local': 'Infraestructura y energía', 'cloud': 'Pago por uso'}
{'criterio': 'Disponibilidad offline', 'local': 'Sí', 'cloud': 'No'}


## 8. Router determinístico cloud/local

El router no “razona” ni es un agente. Aplica reglas explícitas y auditables:

1. una tarea con datos sensibles prefiere local si el modelo local alcanza;
2. la política inicial **prefiere cloud** para tool calling o complejidad alta, aunque el modelo local soporte tools;
3. si el provider elegido no está disponible, se aplica fallback local con revisión humana;
4. la decisión devuelve también su motivo.

La regla 2 es una decisión conservadora de producto, no una afirmación de que Qwen 3.5 carezca de tool calling. Más adelante puede reemplazarse por una política basada en resultados de evaluación.


### 8.1 Probar la política de routing como función pura

Antes de conectarla con providers reales, una política de routing debe poder probarse con una tabla de decisiones. La función recibe señales explícitas y devuelve una decisión explicable; no llama modelos ni depende de la red.

In [24]:

from __future__ import annotations

from dataclasses import asdict, dataclass

@dataclass(frozen=True)
class RoutingDecision:
    provider: str
    reason: str
    requires_human_review: bool = False

    def to_dict(self) -> dict:
        return asdict(self)

SENSITIVE_TERMS = {
    "dni", "documento", "salario", "sueldo", "contraseña",
    "password", "historia clínica", "tarjeta", "cuenta bancaria",
}

def contains_sensitive_data(text: str) -> bool:
    lowered = text.lower()
    return any(term in lowered for term in SENSITIVE_TERMS)

def choose_provider_for_task(
    *,
    text: str,
    needs_tools: bool,
    complexity: str = "low",
    local_available: bool = True,
    cloud_available: bool = True,
) -> RoutingDecision:
    if complexity not in {"low", "medium", "high"}:
        raise ValueError("complexity debe ser low, medium o high")

    sensitive = contains_sensitive_data(text)

    if sensitive:
        if local_available:
            requires_review = needs_tools or complexity == "high"
            reason = (
                "datos sensibles; validar capacidad local y mantener revision humana"
                if requires_review
                else "datos sensibles y capacidad local suficiente"
            )
            return RoutingDecision(
                "local", reason, requires_human_review=requires_review
            )
        if cloud_available:
            return RoutingDecision(
                "cloud",
                "local no disponible; autorizacion explicita requerida antes de enviar datos",
                requires_human_review=True,
            )
        raise RuntimeError("No hay providers disponibles")

    if needs_tools or complexity == "high":
        if cloud_available:
            return RoutingDecision("cloud", "requiere tools o razonamiento complejo")
        if local_available:
            return RoutingDecision(
                "local",
                "cloud no disponible; fallback con revisi?n humana",
                requires_human_review=True,
            )
        raise RuntimeError("No hay providers disponibles")

    if local_available:
        return RoutingDecision("local", "tarea simple; prioriza costo y disponibilidad local")
    if cloud_available:
        return RoutingDecision("cloud", "local no disponible")
    raise RuntimeError("No hay providers disponibles")


In [25]:
routing_cases = [
    {
        "name": "sensible_simple",
        "kwargs": dict(text="El mensaje incluye un DNI", needs_tools=False, complexity="low"),
        "expected": ("local", False),
    },
    {
        "name": "sensible_con_tools",
        "kwargs": dict(
            text="Buscar legajo asociado a un DNI",
            needs_tools=True,
            complexity="medium",
        ),
        "expected": ("local", True),
    },
    {
        "name": "requiere_tools",
        "kwargs": dict(text="Buscar política", needs_tools=True, complexity="medium"),
        "expected": ("cloud", False),
    },
    {
        "name": "cloud_caido",
        "kwargs": dict(
            text="Resolver tarea compleja",
            needs_tools=True,
            complexity="high",
            cloud_available=False,
        ),
        "expected": ("local", True),
    },
]

routing_preview = []
for case in routing_cases:
    decision = choose_provider_for_task(**case["kwargs"])
    assert (decision.provider, decision.requires_human_review) == case["expected"]
    routing_preview.append({"case": case["name"], **decision.to_dict()})

try:
    choose_provider_for_task(text="x", needs_tools=False, complexity="unknown")
except ValueError as exc:
    invalid_complexity = str(exc)

{"decisiones": routing_preview, "borde": invalid_complexity}

{'decisiones': [{'case': 'sensible_simple',
   'provider': 'local',
   'reason': 'datos sensibles y capacidad local suficiente',
   'requires_human_review': False},
  {'case': 'sensible_con_tools',
   'provider': 'local',
   'reason': 'datos sensibles; validar capacidad local y mantener revision humana',
   'requires_human_review': True},
  {'case': 'requiere_tools',
   'provider': 'cloud',
   'reason': 'requiere tools o razonamiento complejo',
   'requires_human_review': False},
  {'case': 'cloud_caido',
   'provider': 'local',
   'reason': 'cloud no disponible; fallback con revisi?n humana',
   'requires_human_review': True}],
 'borde': 'complexity debe ser low, medium o high'}

### 8.2 Persistir una política ya verificada

El módulo conserva la función pura para que pueda reutilizarse desde una API, un agente o un test sin duplicar reglas.

In [26]:
%%writefile ai_agent_project/src/ai_agent_course/routing.py
from __future__ import annotations

from dataclasses import asdict, dataclass

@dataclass(frozen=True)
class RoutingDecision:
    provider: str
    reason: str
    requires_human_review: bool = False

    def to_dict(self) -> dict:
        return asdict(self)

SENSITIVE_TERMS = {
    "dni", "documento", "salario", "sueldo", "contraseña",
    "password", "historia clínica", "tarjeta", "cuenta bancaria",
}

def contains_sensitive_data(text: str) -> bool:
    lowered = text.lower()
    return any(term in lowered for term in SENSITIVE_TERMS)

def choose_provider_for_task(
    *,
    text: str,
    needs_tools: bool,
    complexity: str = "low",
    local_available: bool = True,
    cloud_available: bool = True,
) -> RoutingDecision:
    if complexity not in {"low", "medium", "high"}:
        raise ValueError("complexity debe ser low, medium o high")

    sensitive = contains_sensitive_data(text)

    if sensitive:
        if local_available:
            requires_review = needs_tools or complexity == "high"
            reason = (
                "datos sensibles; validar capacidad local y mantener revision humana"
                if requires_review
                else "datos sensibles y capacidad local suficiente"
            )
            return RoutingDecision(
                "local", reason, requires_human_review=requires_review
            )
        if cloud_available:
            return RoutingDecision(
                "cloud",
                "local no disponible; autorizacion explicita requerida antes de enviar datos",
                requires_human_review=True,
            )
        raise RuntimeError("No hay providers disponibles")

    if needs_tools or complexity == "high":
        if cloud_available:
            return RoutingDecision("cloud", "requiere tools o razonamiento complejo")
        if local_available:
            return RoutingDecision(
                "local",
                "cloud no disponible; fallback con revision humana",
                requires_human_review=True,
            )
        raise RuntimeError("No hay providers disponibles")

    if local_available:
        return RoutingDecision("local", "tarea simple; prioriza costo y disponibilidad local")
    if cloud_available:
        return RoutingDecision("cloud", "local no disponible")
    raise RuntimeError("No hay providers disponibles")


Overwriting ai_agent_project/src/ai_agent_course/routing.py


In [27]:
from ai_agent_course.routing import choose_provider_for_task

routing_cases = [
    {
        "name": "clasificación privada",
        "text": "Clasificá este reclamo que contiene un DNI.",
        "needs_tools": False,
        "complexity": "low",
    },
    {
        "name": "consulta con tool",
        "text": "Buscá la política vigente y citá la fuente.",
        "needs_tools": True,
        "complexity": "medium",
    },
    {
        "name": "resumen simple",
        "text": "Resumí esta nota pública.",
        "needs_tools": False,
        "complexity": "low",
    },
]

routing_rows = []
for item in routing_cases:
    decision = choose_provider_for_task(
        text=item["text"],
        needs_tools=item["needs_tools"],
        complexity=item["complexity"],
        local_available=True,
        cloud_available=True,
    )
    routing_rows.append({"caso": item["name"], **decision.to_dict()})

show_table(routing_rows)

{'caso': 'clasificación privada', 'provider': 'local', 'reason': 'datos sensibles y capacidad local suficiente', 'requires_human_review': False}
{'caso': 'consulta con tool', 'provider': 'cloud', 'reason': 'requiere tools o razonamiento complejo', 'requires_human_review': False}
{'caso': 'resumen simple', 'provider': 'local', 'reason': 'tarea simple; prioriza costo y disponibilidad local', 'requires_human_review': False}


### Falla del provider preferido

Una política útil también define qué hacer cuando el provider elegido no responde. El fallback no debe ser silencioso si cambia el nivel de privacidad o capacidad.

In [28]:
fallback_examples = [
    choose_provider_for_task(
        text="Procesá este salario individual.",
        needs_tools=False,
        complexity="low",
        local_available=False,
        cloud_available=True,
    ).to_dict(),
    choose_provider_for_task(
        text="Usá una tool para actualizar el sistema.",
        needs_tools=True,
        complexity="high",
        local_available=True,
        cloud_available=False,
    ).to_dict(),
]
show_table(fallback_examples)

{'provider': 'cloud', 'reason': 'local no disponible; autorizacion explicita requerida antes de enviar datos', 'requires_human_review': True}
{'provider': 'local', 'reason': 'cloud no disponible; fallback con revision humana', 'requires_human_review': True}


## 9. Tests sin red

Los tests verifican contratos y políticas, no la disponibilidad de un servicio externo. Por eso usan providers simulados.

### 9.1 Persistir los tests de regresión

Estos tests fijan el contrato que deberán respetar las clases siguientes: routing explicable y providers intercambiables.

In [29]:
%%writefile ai_agent_project/tests/test_class04.py
import asyncio

from ai_agent_course.local_models import SimulatedAgentProvider
from ai_agent_course.routing import choose_provider_for_task


def test_local_route_for_sensitive_simple_task():
    decision = choose_provider_for_task(
        text="El mensaje incluye un DNI",
        needs_tools=False,
        complexity="low",
        local_available=True,
        cloud_available=True,
    )
    assert decision.provider == "local"


def test_sensitive_tool_task_never_escalates_silently():
    decision = choose_provider_for_task(
        text="Buscar el legajo asociado a este DNI",
        needs_tools=True,
        complexity="medium",
        local_available=True,
        cloud_available=True,
    )
    assert decision.provider == "local"
    assert decision.requires_human_review is True


def test_cloud_route_for_tools():
    decision = choose_provider_for_task(
        text="Buscar una política",
        needs_tools=True,
        complexity="medium",
        local_available=True,
        cloud_available=True,
    )
    assert decision.provider == "cloud"


def test_provider_returns_common_contract():
    provider = SimulatedAgentProvider(
        name="test",
        model="test-model",
        latency_ms=1,
        supports_tools=True,
        structured_reliability=1.0,
    )
    result = asyncio.run(provider.agenerate("Devolvé SOLO JSON con category y confidence"))
    assert result.provider == "test"
    assert result.model == "test-model"
    assert result.text


def test_native_tool_call_has_structured_arguments():
    provider = SimulatedAgentProvider(
        name="test",
        model="test-model",
        latency_ms=1,
        supports_tools=True,
        structured_reliability=1.0,
    )
    tool = {
        "type": "function",
        "function": {
            "name": "buscar_politica",
            "description": "Busca una política.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
            },
        },
    }
    result = asyncio.run(
        provider.achat_with_tools("Buscá vacaciones", tools=[tool])
    )
    call = result["tool_calls"][0]["function"]
    assert call["name"] == "buscar_politica"
    assert call["arguments"]["query"] == "vacaciones"


Overwriting ai_agent_project/tests/test_class04.py


In [30]:
# Ejecución liviana dentro de la notebook
from ai_agent_course.local_models import SimulatedAgentProvider

test_provider = SimulatedAgentProvider(
    name="test",
    model="test-model",
    latency_ms=1,
    supports_tools=True,
    structured_reliability=1.0,
)
test_result = await test_provider.agenerate("Respondé una prueba")
assert test_result.text

test_tool_result = await test_provider.achat_with_tools(
    "Buscá vacaciones",
    tools=[POLICY_TOOL],
)
assert test_tool_result["tool_calls"][0]["function"]["name"] == "buscar_politica"

assert choose_provider_for_task(
    text="contiene contraseña",
    needs_tools=False,
    complexity="low",
).provider == "local"
print("✓ Tests esenciales de la Clase 4: OK")


✓ Tests esenciales de la Clase 4: OK


## 10. Caso integrador

Seleccionamos provider, ejecutamos la tarea y devolvemos una traza mínima. Esta función anticipa el patrón que más adelante usará un agente, pero todavía conserva decisiones determinísticas.

In [31]:
async def execute_routed_task(task: dict) -> dict:
    decision = choose_provider_for_task(
        text=task["prompt"],
        needs_tools=task["needs_tools"],
        complexity=task["complexity"],
        local_available=True,
        cloud_available=True,
    )
    provider = providers[decision.provider]
    tool_calls = []

    if task["needs_tools"] and hasattr(provider, "achat_with_tools"):
        tool_result = await provider.achat_with_tools(
            task["prompt"], tools=[POLICY_TOOL], temperature=0
        )
        tool_calls = tool_result.get("tool_calls", [])
        response = tool_result.get("content", "")
        model = tool_result.get("model")
        latency_ms = tool_result.get("latency_ms")
        execution_mode = "native_tool_call"
    else:
        result = await provider.agenerate(task["prompt"], temperature=0)
        response = result.text
        model = result.model
        latency_ms = result.latency_ms
        execution_mode = (
            "prompted_tool_fallback" if task["needs_tools"] else "generation"
        )

    return {
        "task_id": task["task_id"],
        "selected_provider": decision.provider,
        "routing_reason": decision.reason,
        "requires_human_review": decision.requires_human_review,
        "execution_mode": execution_mode,
        "model": model,
        "latency_ms": latency_ms,
        "response": response,
        "tool_calls": tool_calls,
    }

integrated_tasks = [
    {
        "task_id": "T-001",
        "prompt": "Clasificá este texto con DNI como consulta de RRHH.",
        "needs_tools": False,
        "complexity": "low",
    },
    {
        "task_id": "T-002",
        "prompt": (
            'Seleccioná una herramienta. Devolvé SOLO JSON con tool y arguments. '
            'Herramienta disponible: buscar_politica(query). Consulta: vacaciones.'
        ),
        "needs_tools": True,
        "complexity": "medium",
    },
]

integrated_results = []
for task in integrated_tasks:
    integrated_results.append(await execute_routed_task(task))

show_table(integrated_results)

{'task_id': 'T-001', 'selected_provider': 'local', 'routing_reason': 'datos sensibles y capacidad local suficiente', 'requires_human_review': False, 'execution_mode': 'generation', 'model': 'qwen3.5:4b', 'latency_ms': 36.13, 'response': 'Respuesta de ollama-simulado: Clasificá este texto con DNI como consulta de RRHH.', 'tool_calls': []}
{'task_id': 'T-002', 'selected_provider': 'cloud', 'routing_reason': 'requiere tools o razonamiento complejo', 'requires_human_review': False, 'execution_mode': 'native_tool_call', 'model': 'gemini-3.1-flash-lite', 'latency_ms': 71.12, 'response': '', 'tool_calls': [{'function': {'name': 'buscar_politica', 'arguments': {'query': 'vacaciones'}}}]}


## 11. Checkpoint

El artefacto registra configuración segura, estado de Ollama, benchmark, routing y ejecución integrada. No guarda API keys.

In [32]:
checkpoint = {
    "class": 4,
    "cloud_model": GEMINI_MODEL,
    "local_model": OLLAMA_MODEL,
    "ollama_status": ollama_status,
    "benchmark": benchmark_rows,
    "summary": summary_rows,
    "native_tool_call": native_tool_check,
    "routing_examples": routing_rows,
    "integrated_results": integrated_results,
    "notes": [
        "Los resultados simulados validan plumbing y contratos.",
        "La calidad real debe medirse con providers reales y múltiples ejecuciones.",
        "El JSON por prompt y tool calling nativo se registran como evidencias distintas.",
    ],
}

write_file(
    ARTIFACTS_DIR / "class04_model_benchmark.json",
    json.dumps(checkpoint, ensure_ascii=False, indent=2),
)

assert checkpoint["cloud_model"] == "gemini-3.1-flash-lite"
assert all("provider" in row for row in summary_rows)
print("✓ Checkpoint Clase 4 OK")

✓ artifacts/class04_model_benchmark.json
✓ Checkpoint Clase 4 OK


## Cierre

La decisión cloud/local no se resuelve con una preferencia tecnológica. Se resuelve con una política explícita y evidencia:

- ¿la tarea requiere tools o structured output confiable?
- ¿el soporte declarado funciona bien en nuestros casos reales?
- ¿qué datos se procesan?
- ¿qué latencia y concurrencia se necesitan?
- ¿qué ocurre cuando un provider falla?
- ¿cómo se mide la calidad?

También quedó establecida una distinción que será importante durante todo el curso: **generar un JSON que menciona una tool no es lo mismo que recibir un `tool_call` nativo del runtime**.

En la Clase 5 la misma idea de contratos se aplica a **embeddings, chunking y retrieval**.


In [33]:
_checkpoint_dir = PROJECT_DIR / "artifacts" / "checkpoints"
_checkpoint_dir.mkdir(parents=True, exist_ok=True)
_checkpoint = {"class": 4, "canonical_model": "gemini-3.1-flash-lite", "increment": 'Ollama, benchmark y routing cloud/local', "default_mode": "simulated_or_local", "next_class": 5}
_checkpoint_path = _checkpoint_dir / "class04_checkpoint.json"
_checkpoint_path.write_text(json.dumps(_checkpoint, ensure_ascii=False, indent=2), encoding="utf-8")
print("Checkpoint transversal:", _checkpoint_path.relative_to(PROJECT_DIR))

Checkpoint transversal: artifacts/checkpoints/class04_checkpoint.json


## Continuidad del proyecto

El incremento queda guardado dentro de `ai_agent_project/` para que la siguiente clase lo importe y lo extienda.